In [46]:
import pandas as pd
from pathlib import Path
import numpy as np
np.set_printoptions(suppress=True)

In [31]:
# paths = (Path().cwd() / 'data').iterdir()
# columns = ['Date','1 Mo','2 Mo','3 Mo','6 Mo','1 Yr','2 Yr','3 Yr','5 Yr','7 Yr','10 Yr','20 Yr','30 Yr']
# for path in paths:
#     if path.suffix == '.csv' and path.name.startswith('par-yield'):
#         df = pd.read_csv(path, usecols=columns)
#         df['Date'] = pd.to_datetime(df['Date'])
#         df.set_index('Date', inplace=True)
#         all_data.append(df)

# combined_data = pd.concat(all_data)
# combined_data.dropna(inplace=True)
# combined_data

In [32]:
df = pd.read_csv("data/par-yield-curve-rates-2020-2023.csv")
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
df.head()

,1 mo,2 mo,3 mo,4 mo,6 mo,1 yr,2 yr,3 yr,5 yr,7 yr,10 yr,20 yr,30 yr
date,,,,,,,,,,,,,
2020-01-02,1.53,1.55,1.54,NaN,1.57,1.56,1.58,1.59,1.67,1.79,1.88,2.19,2.33
2020-01-03,1.52,1.55,1.52,NaN,1.55,1.55,1.53,1.54,1.59,1.71,1.80,2.11,2.26
2020-01-06,1.54,1.54,1.56,NaN,1.56,1.54,1.54,1.56,1.61,1.72,1.81,2.13,2.28
2020-01-07,1.52,1.53,1.54,NaN,1.56,1.53,1.54,1.55,1.62,1.74,1.83,2.16,2.31
2020-01-08,1.50,1.53,1.54,NaN,1.56,1.55,1.58,1.61,1.67,1.78,1.87,2.21,2.35


In [33]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1001 entries, 2020-01-02 to 2023-12-29
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   1 mo    1001 non-null   float64
 1   2 mo    1001 non-null   float64
 2   3 mo    1001 non-null   float64
 3   4 mo    300 non-null    float64
 4   6 mo    1001 non-null   float64
 5   1 yr    1001 non-null   float64
 6   2 yr    1001 non-null   float64
 7   3 yr    1001 non-null   float64
 8   5 yr    1001 non-null   float64
 9   7 yr    1001 non-null   float64
 10  10 yr   1001 non-null   float64
 11  20 yr   1001 non-null   float64
 12  30 yr   1001 non-null   float64
dtypes: float64(13)
memory usage: 109.5 KB


In [34]:
df.dropna(axis=1, inplace=True)

In [35]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1001 entries, 2020-01-02 to 2023-12-29
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   1 mo    1001 non-null   float64
 1   2 mo    1001 non-null   float64
 2   3 mo    1001 non-null   float64
 3   6 mo    1001 non-null   float64
 4   1 yr    1001 non-null   float64
 5   2 yr    1001 non-null   float64
 6   3 yr    1001 non-null   float64
 7   5 yr    1001 non-null   float64
 8   7 yr    1001 non-null   float64
 9   10 yr   1001 non-null   float64
 10  20 yr   1001 non-null   float64
 11  30 yr   1001 non-null   float64
dtypes: float64(12)
memory usage: 101.7 KB


In [48]:
from sklearn.decomposition import PCA
from numpy.linalg import eigvals

pca = PCA(n_components=1)
principal_components = pca.fit_transform(df)
pca.explained_variance_ratio_

array([0.96724723])

In [51]:
# Wykres świecowy (candlestick) dla wybranego tenor'u (domyślnie '10 yr').
# Używa już istniejącego df z indeksem dat (datetime).
tenor = '5 yr'  # zmień na np. '1 mo', '5 yr' itp.

# znajdź kolumnę ignorując wielkość liter/spacje
cols_map = {c.lower(): c for c in df.columns}
key = tenor.lower()
if key not in cols_map:
    raise KeyError(f"Nie znaleziono tenor'a '{tenor}'. Dostępne kolumny: {list(df.columns)}")

series = df[cols_map[key]]

# zgrupuj do OHLC (tu przykładowo resampling tygodniowy). Możesz zmienić 'W' na 'D','M' itp.
ohlc = series.resample('W').agg(['first', 'max', 'min', 'last']).dropna()
ohlc.columns = ['open', 'high', 'low', 'close']

# wykres ze plotly
import plotly.graph_objects as go

fig = go.Figure(data=[go.Candlestick(
    x=ohlc.index,
    open=ohlc['open'],
    high=ohlc['high'],
    low=ohlc['low'],
    close=ohlc['close'],
    increasing_line_color='green',
    decreasing_line_color='red'
)])
fig.update_layout(
    title=f'Candlestick - {cols_map[key]} (resampled weekly)',
    xaxis_title='Date',
    yaxis_title='Rate',
    xaxis_rangeslider_visible=False
)
fig.show()